# Normalizacao do Relatorio de Receitas do Sistema ATUA (03/2026)

Este notebook le o arquivo `Relatorio_Receitas_Sistema-ATUA_03.xls` (sistema ATUA, GSL Logistica) e converte para o layout do fechamento SAGI (`FECHAMENTO_ODBC_2026_03.xlsx`).

Cada linha do arquivo e um CTRC (Conhecimento de Transporte Rodoviario de Cargas) — ou seja, uma receita de frete faturada. No SAGI, a GSL esta na divisao **2.4 TRANSMOVE** (lado receita), com tres filiais ativas neste relatorio:

- GSL PRUDENTE -> 2.4.1 PRESIDENTE PRUDENTE -> 2.4.1.1 TRANSPORTE
- GSL DOURADOS -> 2.4.2 DOURADOS -> 2.4.2.1 TRANSPORTE
- GSL MARINGA PR -> 2.4.3 MARINGA -> 2.4.3.1 TRANSPORTE

O Plano de Contas de todas as linhas e **5.7.1 FRETES** (receita de frete proprio).

In [7]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

REFS_DIR = Path("../../02-Referencias")
ATUA_DIR = REFS_DIR / "ATUA"
ARQUIVO_ENTRADA  = ATUA_DIR / "Relatorio_Receitas_Sistema-ATUA_03.xls"
ARQUIVO_MODELO   = REFS_DIR / "FECHAMENTO_ODBC_2026_03.xlsx"
ARQUIVO_SAIDA    = ATUA_DIR / "ATUA_receitas_fechamento_03-2026.xlsx"

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f"Arquivo ATUA Receitas nao encontrado: {ARQUIVO_ENTRADA.resolve()}")

# O arquivo possui pequena corrupcao interna — lemos com ignore_workbook_corruption
df_receitas = pd.read_excel(
    ARQUIVO_ENTRADA,
    sheet_name=0,
    header=0,
    dtype=object,
    engine_kwargs={"ignore_workbook_corruption": True},
)

print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Linhas lidas: {len(df_receitas)}")
print(f"Colunas: {df_receitas.columns.tolist()}")
print()
df_receitas.head(5)

Entrada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\Relatorio_Receitas_Sistema-ATUA_03.xls
Linhas lidas: 138
Colunas: ['nr_ctrc', 'dt_emissao', 'nm_pessoa_filial', 'nm_pessoa_destinatario', 'nm_cidade_destinatario', 'nm_pessoa_motorista', 'cd_pessoa_proprietario_veiculo_posse', 'vl_frete_empresa', 'vl_frete_motorista']



,nr_ctrc,dt_emissao,nm_pessoa_filial,nm_pessoa_destinatario,nm_cidade_destinatario,nm_pessoa_motorista,cd_pessoa_proprietario_veiculo_posse,vl_frete_empresa,vl_frete_motorista
0,150,02/03/26,GSL DOURADOS,GERDAU ACOS LONGOS SA,ARACARIGUAMA,FELIPE TESTE CAMOICO,7960,12105,8715.6
1,850,02/03/26,GSL PRUDENTE,G3S COMERCIO E INDUSTRIA DE FERRO E ACO LTDA.,PRESIDENTE PRUDENTE,JULIANO VIEIRA DE SANTANA,54,600,0
2,851,02/03/26,GSL PRUDENTE,GV DO BRASIL INDUSTRIA E COMERCIO DE ACO LTDA,PINDAMONHANGABA,ANTONIO SARAFIM BARBOZA,4574,10038,7695.8
3,151,03/03/26,GSL DOURADOS,GERDAU ACOS LONGOS SA,ARACARIGUAMA,SANDOVAL MARQUES DE OLIVEIRA,10438,13121.25,9447.3
4,152,03/03/26,GSL DOURADOS,GERDAU ACOS LONGOS S.A,RIO DE JANEIRO,SILVIO FERNANDO BRAZ MEIRA,2888,15593.25,12841.5


## Mapeamento de Centro de Custo (Receita)

O campo `nm_pessoa_filial` no ATUA identifica qual filial GSL emitiu o frete. O mapeamento para a hierarquia SAGI e:

| nm_pessoa_filial | n2     | n3     | n4       | Descricoes                                 |
|------------------|--------|--------|----------|--------------------------------------------|
| GSL PRUDENTE     | 2.4    | 2.4.1  | 2.4.1.1  | TRANSMOVE GSL > PRESIDENTE PRUDENTE > TRANSPORTE |
| GSL DOURADOS     | 2.4    | 2.4.2  | 2.4.2.1  | TRANSMOVE GSL > DOURADOS > TRANSPORTE      |
| GSL MARINGA PR   | 2.4    | 2.4.3  | 2.4.3.1  | TRANSMOVE GSL > MARINGA > TRANSPORTE       |

In [8]:
def _str(v) -> str:
    if pd.isna(v):
        return ""
    return str(v).strip()

MAPA_FILIAL_RECEITA = {
    "GSL PRUDENTE": {
        "n3_cod":  "2.4.1",
        "n3_desc": "PRESIDENTE PRUDENTE",
        "n4_cod":  "2.4.1.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL PRUDENTE",
    },
    "GSL DOURADOS": {
        "n3_cod":  "2.4.2",
        "n3_desc": "DOURADOS",
        "n4_cod":  "2.4.2.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL DOURADOS",
    },
    "GSL MARINGA PR": {
        "n3_cod":  "2.4.3",
        "n3_desc": "MARINGA",
        "n4_cod":  "2.4.3.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL MARINGA",
    },
}

def mapear_cc_receita(nm_filial) -> dict | None:
    """Retorna hierarquia SAGI completa para a filial, ou None se nao mapeada."""
    chave = _str(nm_filial)
    info  = MAPA_FILIAL_RECEITA.get(chave)
    if not info:
        return None
    return {
        "n1_cod":  "2",
        "n1_desc": "RECEITA",
        "n2_cod":  "2.4",
        "n2_desc": "TRANSMOVE GSL",
        "n3_cod":  info["n3_cod"],
        "n3_desc": info["n3_desc"],
        "n4_cod":  info["n4_cod"],
        "n4_desc": info["n4_desc"],
        "filial_saida": info["filial_saida"],
        "segmento": "TRANSMOVE GSL",
    }

print("Filiais unicas no arquivo:")
print(df_receitas["nm_pessoa_filial"].value_counts(dropna=False).to_string())
print()

filiais_nao_mapeadas = set()
for v in df_receitas["nm_pessoa_filial"].unique():
    chave = _str(v)
    if chave not in MAPA_FILIAL_RECEITA:
        filiais_nao_mapeadas.add(chave)

if filiais_nao_mapeadas:
    print(f"[AVISO] Filiais NAO mapeadas ({len(filiais_nao_mapeadas)}):")
    for f in sorted(filiais_nao_mapeadas):
        print(f"  '{f}'")
else:
    print("[OK] Todas as filiais estao mapeadas.")

Filiais unicas no arquivo:
nm_pessoa_filial
GSL PRUDENTE      58
GSL MARINGA PR    50
GSL DOURADOS      30

[OK] Todas as filiais estao mapeadas.


## Plano de Contas — Fixo: 5.7.1 FRETES

Todas as 138 linhas deste relatorio sao CTRCs (fretes proprios da GSL faturados a clientes). No SAGI, a conta de receita correspondente e **5.7.1 FRETES**.

In [9]:
COD_CONTA_RECEITA  = "5.7.1"
DESC_CONTA_RECEITA = "FRETES"

print(f"Plano de Contas fixo para todas as linhas: {COD_CONTA_RECEITA} {DESC_CONTA_RECEITA}")

Plano de Contas fixo para todas as linhas: 5.7.1 FRETES


## Conversao para o layout FECHAMENTO_ODBC

Regras de mapeamento de colunas:

| Coluna FECHAMENTO_ODBC | Origem ATUA                          | Observacao                                    |
|------------------------|--------------------------------------|-----------------------------------------------|
| filial                 | filial_saida (do mapa CC)            |                                               |
| titulo                 | `CTRC-{nr_ctrc}`                     |                                               |
| credor_forn_cli_func   | nm_pessoa_destinatario               | cliente / tomador do frete                    |
| data_nf                | dt_emissao                           | data de emissao do CTRC                       |
| data_pagamento         | dt_emissao                           | mesma data (nao ha data de pagamento separada) |
| valor_nf               | vl_frete_empresa                     |                                               |
| valor_pago             | vl_frete_empresa                     |                                               |
| valor_conta            | vl_frete_empresa                     |                                               |
| cod_conta              | `5.7.1`                              |                                               |
| conta                  | `FRETES`                             |                                               |
| observacao             | CTRC {nr_ctrc} - Motorista: ... - Destino: ... |                                      |
| Origem                 | `Saida (Aplicacoes)`                 |                                               |
| Sistema                | `ATUA`                               |                                               |
| n1_cod … n4_desc       | MAPA_FILIAL_RECEITA                  |                                               |

In [10]:
def _to_float(v):
    if pd.isna(v):
        return 0.0
    try:
        return float(str(v).strip().replace(",", "."))
    except (ValueError, TypeError):
        return 0.0

def _fmt_brl(v) -> str:
    """Formata float como string no padrao brasileiro (ex: 1.234,56)."""
    try:
        f = _to_float(v)
        return f"{f:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return str(v)

def _fmt_data(v) -> str:
    """Converte 'dd/mm/yy ' (com possivel espaco) para 'dd/mm/aaaa'."""
    s = _str(v)
    if not s:
        return ""
    # Tenta interpretar formatos possiveis
    for fmt in ("%d/%m/%y", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
        try:
            import datetime
            return datetime.datetime.strptime(s, fmt).strftime("%d/%m/%Y")
        except ValueError:
            pass
    return s  # fallback: devolve como veio

# Carrega colunas do modelo para garantir ordem e completude
modelo_cols = pd.read_excel(ARQUIVO_MODELO, nrows=0).columns.tolist()

filiais_sem_mapa = []
linhas_saida = []

for _, row in df_receitas.iterrows():
    filial_raw = _str(row.get("nm_pessoa_filial", ""))
    cc = mapear_cc_receita(filial_raw)

    if cc is None:
        filiais_sem_mapa.append(filial_raw)

    ctrc     = _str(row.get("nr_ctrc", ""))
    dt_emis  = _fmt_data(row.get("dt_emissao", ""))
    valor    = _to_float(row.get("vl_frete_empresa", 0))
    destino  = _str(row.get("nm_cidade_destinatario", ""))
    motorist = _str(row.get("nm_pessoa_motorista", ""))
    cliente  = _str(row.get("nm_pessoa_destinatario", ""))

    nova = {col: "" for col in modelo_cols}

    nova["filial"]               = cc["filial_saida"] if cc else filial_raw
    nova["titulo"]               = f"CTRC-{ctrc}"
    nova["credor_forn_cli_func"] = cliente
    nova["data_nf"]              = dt_emis
    nova["data_pagamento"]       = dt_emis
    nova["valor_nf"]             = _fmt_brl(valor)
    nova["valor_pago"]           = _fmt_brl(valor)
    nova["valor_conta"]          = _fmt_brl(valor)
    nova["cod_conta"]            = COD_CONTA_RECEITA
    nova["conta"]                = DESC_CONTA_RECEITA
    nova["observacao"]           = f"CTRC {ctrc} - Motorista: {motorist} - Destino: {destino}"
    nova["Origem"]               = "Saida (Aplicacoes)"
    nova["Sistema"]              = "ATUA"
    nova["Segmento"]             = cc["segmento"]     if cc else ""
    nova["cod_conta-descr"]      = f"{COD_CONTA_RECEITA} {DESC_CONTA_RECEITA}"
    nova["n1_cod_centro_custo"]  = cc["n1_cod"]       if cc else ""
    nova["n1_centro_custo"]      = cc["n1_desc"]      if cc else ""
    nova["n1_CC"]                = f"{cc['n1_cod']} {cc['n1_desc']}" if cc else ""
    nova["n2_cod_centro_custo"]  = cc["n2_cod"]       if cc else ""
    nova["n2_centro_custo"]      = cc["n2_desc"]      if cc else ""
    nova["n2_CC"]                = f"{cc['n2_cod']} {cc['n2_desc']}" if cc else ""
    nova["n3_cod_centro_custo"]  = cc["n3_cod"]       if cc else ""
    nova["n3_centro_custo"]      = cc["n3_desc"]      if cc else ""
    nova["n3_CC"]                = f"{cc['n3_cod']} {cc['n3_desc']}" if cc else ""
    nova["n4_cod_centro_custo"]  = cc["n4_cod"]       if cc else ""
    nova["n4_centro_custo"]      = cc["n4_desc"]      if cc else ""
    nova["n4_CC"]                = f"{cc['n4_cod']} {cc['n4_desc']}" if cc else ""

    linhas_saida.append(nova)

fechamento_df = pd.DataFrame(linhas_saida, columns=modelo_cols)

print(f"Linhas geradas: {len(fechamento_df)}")
print(f"Filiais sem mapeamento: {len(filiais_sem_mapa)}")
if filiais_sem_mapa:
    print(set(filiais_sem_mapa))
print()
fechamento_df.head(5)

Linhas geradas: 138
Filiais sem mapeamento: 0



,id,Segmento,n1_cod_centro_custo,n1_centro_custo,n1_CC,n2_cod_centro_custo,n2_centro_custo,n2_CC,n3_cod_centro_custo,n3_centro_custo,n3_CC,n4_cod_centro_custo,n4_centro_custo,n4_CC,cod_conta,...,valor_nf,valor_pago,valor_conta,observacao,data_nf,data_pagamento,cod_credor_forn_cli_func,credor_forn_cli_func,Origem,Sistema,Dados auxiliares,Valor Oficial,DE-PARA1,DE-PARA2,CUSTEIO VARIÁVEL
0,,TRANSMOVE GSL,2,RECEITA,2 RECEITA,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.2,DOURADOS,2.4.2 DOURADOS,2.4.2.1,TRANSPORTE,2.4.2.1 TRANSPORTE,5.7.1,...,"12.105,00","12.105,00","12.105,00",CTRC 150 - Motorista: FELIPE TESTE CAMOICO - D...,02/03/2026,02/03/2026,,GERDAU ACOS LONGOS SA,Saida (Aplicacoes),ATUA,,,,,
1,,TRANSMOVE GSL,2,RECEITA,2 RECEITA,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.1,PRESIDENTE PRUDENTE,2.4.1 PRESIDENTE PRUDENTE,2.4.1.1,TRANSPORTE,2.4.1.1 TRANSPORTE,5.7.1,...,"600,00","600,00","600,00",CTRC 850 - Motorista: JULIANO VIEIRA DE SANTAN...,02/03/2026,02/03/2026,,G3S COMERCIO E INDUSTRIA DE FERRO E ACO LTDA.,Saida (Aplicacoes),ATUA,,,,,
2,,TRANSMOVE GSL,2,RECEITA,2 RECEITA,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.1,PRESIDENTE PRUDENTE,2.4.1 PRESIDENTE PRUDENTE,2.4.1.1,TRANSPORTE,2.4.1.1 TRANSPORTE,5.7.1,...,"10.038,00","10.038,00","10.038,00",CTRC 851 - Motorista: ANTONIO SARAFIM BARBOZA ...,02/03/2026,02/03/2026,,GV DO BRASIL INDUSTRIA E COMERCIO DE ACO LTDA,Saida (Aplicacoes),ATUA,,,,,
3,,TRANSMOVE GSL,2,RECEITA,2 RECEITA,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.2,DOURADOS,2.4.2 DOURADOS,2.4.2.1,TRANSPORTE,2.4.2.1 TRANSPORTE,5.7.1,...,"13.121,25","13.121,25","13.121,25",CTRC 151 - Motorista: SANDOVAL MARQUES DE OLIV...,03/03/2026,03/03/2026,,GERDAU ACOS LONGOS SA,Saida (Aplicacoes),ATUA,,,,,
4,,TRANSMOVE GSL,2,RECEITA,2 RECEITA,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.2,DOURADOS,2.4.2 DOURADOS,2.4.2.1,TRANSPORTE,2.4.2.1 TRANSPORTE,5.7.1,...,"15.593,25","15.593,25","15.593,25",CTRC 152 - Motorista: SILVIO FERNANDO BRAZ MEI...,03/03/2026,03/03/2026,,GERDAU ACOS LONGOS S.A,Saida (Aplicacoes),ATUA,,,,,


## Salvando o arquivo de saida

In [11]:
from openpyxl.styles import Font

arquivo_saida_exec = ARQUIVO_SAIDA

try:
    with pd.ExcelWriter(arquivo_saida_exec, engine="openpyxl") as writer:
        sheet_name = "ATUA_receitas"
        fechamento_df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.book[sheet_name]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font
    print(f"Arquivo gerado: {arquivo_saida_exec.resolve()}")
    print(f"Linhas gravadas: {len(fechamento_df)}")
except PermissionError:
    import datetime
    ts = datetime.datetime.now().strftime("%H%M%S")
    alt = arquivo_saida_exec.with_stem(f"{arquivo_saida_exec.stem}_{ts}")
    with pd.ExcelWriter(alt, engine="openpyxl") as writer:
        sheet_name = "ATUA_receitas"
        fechamento_df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.book[sheet_name]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font
    print(f"[AVISO] Arquivo principal em uso. Salvo como: {alt.resolve()}")
    print(f"Linhas gravadas: {len(fechamento_df)}")

print()

if filiais_sem_mapa:
    print(f"[PENDENTE] {len(filiais_sem_mapa)} linha(s) com filial nao mapeada:")
    for f in sorted(set(filiais_sem_mapa)):
        n = filiais_sem_mapa.count(f)
        print(f"  '{f}' ({n} ocorrencia(s))")
else:
    print("[OK] Todas as filiais foram mapeadas. Nenhum item pendente.")

Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\ATUA_receitas_fechamento_03-2026.xlsx
Linhas gravadas: 138

[OK] Todas as filiais foram mapeadas. Nenhum item pendente.
